In [2]:
# ============================================================
# FINTECH PAYMENT FAILURE PREDICTION DATASET GENERATOR
# ------------------------------------------------------------
# Creates:
#   - 10,000 synthetic payment transactions
#   - ~39 columns
#   - ~10% payment failure rate
#   - Deliberately dirty data for Alteryx cleaning/validation
#
# Output:
#   fintech_payment_failure_dirty.csv
#
# Requirements:
#   pip install pandas numpy
# ============================================================

import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

N_ROWS = 10_000
SEED = 42
TARGET_FAILURE_RATE = 0.10

OUTPUT_FILE = "fintech_payment_failure_dirty.csv"


# ============================================================
# MAIN FUNCTION
# ============================================================

def generate_dataset(
    n=10_000,
    seed=42,
    target_failure_rate=0.10
):
    rng = np.random.default_rng(seed)

    # ========================================================
    # MASTER DATA
    # ========================================================

    banks = [
        "HDFC Bank",
        "ICICI Bank",
        "Axis Bank",
        "SBI",
        "Kotak Mahindra Bank",
        "Yes Bank",
        "IndusInd Bank",
        "IDFC First Bank",
        "Federal Bank",
        "Bank of Baroda"
    ]

    gateways = [
        "Razorpay",
        "Cashfree",
        "PayU",
        "BillDesk",
        "CCAvenue"
    ]

    cities_states = [
        ("Ahmedabad", "Gujarat"),
        ("Mumbai", "Maharashtra"),
        ("Pune", "Maharashtra"),
        ("Bengaluru", "Karnataka"),
        ("Hyderabad", "Telangana"),
        ("Chennai", "Tamil Nadu"),
        ("Delhi", "Delhi"),
        ("New Delhi", "Delhi"),
        ("Kolkata", "West Bengal"),
        ("Jaipur", "Rajasthan"),
        ("Lucknow", "Uttar Pradesh"),
        ("Surat", "Gujarat"),
        ("Vadodara", "Gujarat"),
        ("Indore", "Madhya Pradesh"),
        ("Noida", "Uttar Pradesh"),
        ("Gurugram", "Haryana"),
        ("Kochi", "Kerala"),
        ("Chandigarh", "Chandigarh")
    ]

    merchant_categories = [
        "Food",
        "Travel",
        "E-commerce",
        "Utilities",
        "Gaming",
        "Education",
        "Healthcare",
        "Insurance",
        "Entertainment",
        "Groceries"
    ]

    payment_methods = [
        "UPI",
        "Credit Card",
        "Debit Card",
        "Net Banking",
        "Wallet"
    ]

    card_networks = [
        "Visa",
        "Mastercard",
        "RuPay"
    ]

    devices = [
        "Android",
        "iOS",
        "Web"
    ]

    networks = [
        "4G",
        "WiFi",
        "3G"
    ]

    app_versions = [
        "3.8.1",
        "3.9.0",
        "4.0.0",
        "4.0.1",
        "4.1.0",
        "4.1.2",
        "4.2.0"
    ]

    # ========================================================
    # CUSTOMER + MERCHANT UNIVERSE
    # ========================================================

    n_customers = max(2_000, int(n * 0.25))
    n_merchants = max(500, int(n * 0.05))

    customer_ids = np.array([
        f"CUST{100000 + i}"
        for i in range(n_customers)
    ])

    merchant_ids = np.array([
        f"MER{200000 + i}"
        for i in range(n_merchants)
    ])

    customer_idx = rng.integers(
        0,
        n_customers,
        n
    )

    merchant_idx = rng.integers(
        0,
        n_merchants,
        n
    )

    # ========================================================
    # CUSTOMER MASTER DATA
    # These represent information available BEFORE the
    # current transaction.
    # ========================================================

    first_names = [
        "Aarav", "Vivaan", "Aditya", "Arjun", "Rohan",
        "Rahul", "Karan", "Vikram", "Neha", "Priya",
        "Ananya", "Isha", "Aditi", "Riya", "Simran",
        "Pooja", "Sneha", "Meera", "Kavya", "Nisha"
    ]

    last_names = [
        "Shah", "Patel", "Mehta", "Desai", "Sharma",
        "Verma", "Gupta", "Singh", "Iyer", "Reddy",
        "Nair", "Khan", "Joshi", "Jain", "Agarwal"
    ]

    customer_first = rng.choice(
        first_names,
        n_customers
    )

    customer_last = rng.choice(
        last_names,
        n_customers
    )

    customer_master_names = np.array([
        f"{first} {last}"
        for first, last
        in zip(customer_first, customer_last)
    ])

    # --------------------------------------------------------
    # Age + DOB
    # --------------------------------------------------------

    ages = rng.integers(
        18,
        81,
        n_customers
    )

    dob_dates = []

    for age in ages:
        dob = (
            datetime(2026, 1, 1)
            - timedelta(
                days=int(
                    age * 365.25 +
                    rng.integers(0, 365)
                )
            )
        )
        dob_dates.append(dob)

    # --------------------------------------------------------
    # Account age
    # --------------------------------------------------------

    account_age_days_master = rng.integers(
        30,
        3651,
        n_customers
    )

    # --------------------------------------------------------
    # Customer risk segment
    # Used to generate realistic history
    # --------------------------------------------------------

    customer_risk_segment = rng.choice(
        ["LowRisk", "MediumRisk", "HighRisk"],
        n_customers,
        p=[0.55, 0.35, 0.10]
    )

    # --------------------------------------------------------
    # Past transaction count
    # --------------------------------------------------------

    transaction_lambda = np.select(
        [
            customer_risk_segment == "LowRisk",
            customer_risk_segment == "MediumRisk",
            customer_risk_segment == "HighRisk"
        ],
        [
            6.5,
            10.0,
            14.0
        ]
    )

    past_txn_count_master = rng.poisson(
        transaction_lambda
    )

    past_txn_count_master = np.clip(
        past_txn_count_master,
        0,
        60
    )

    # --------------------------------------------------------
    # Historical failure rate
    # --------------------------------------------------------

    beta_a = np.select(
        [
            customer_risk_segment == "LowRisk",
            customer_risk_segment == "MediumRisk",
            customer_risk_segment == "HighRisk"
        ],
        [
            1.5,
            2.0,
            3.0
        ]
    )

    beta_b = np.select(
        [
            customer_risk_segment == "LowRisk",
            customer_risk_segment == "MediumRisk",
            customer_risk_segment == "HighRisk"
        ],
        [
            35,
            18,
            10
        ]
    )

    past_failure_rate_master = rng.beta(
        beta_a,
        beta_b
    )

    past_failure_rate_master = np.clip(
        past_failure_rate_master,
        0,
        0.50
    )

    # --------------------------------------------------------
    # Historical average transaction amount
    # --------------------------------------------------------

    avg_txn_amount_master = np.exp(
        rng.normal(
            np.log(1200),
            0.65,
            n_customers
        )
    )

    avg_txn_amount_master = np.clip(
        avg_txn_amount_master,
        100,
        25_000
    )

    # --------------------------------------------------------
    # Balance bucket
    # --------------------------------------------------------

    account_balance_master = rng.choice(
        ["Low", "Medium", "High"],
        n_customers,
        p=[0.20, 0.55, 0.25]
    )

    # ========================================================
    # CURRENT TRANSACTION DATA
    # ========================================================

    start_date = pd.Timestamp("2026-09-01")

    txn_dt = (
        start_date
        + pd.to_timedelta(
            rng.integers(
                0,
                30 * 24 * 60,
                n
            ),
            unit="m"
        )
    )

    # --------------------------------------------------------
    # Location
    # --------------------------------------------------------

    city_state_idx = rng.integers(
        0,
        len(cities_states),
        n
    )

    cities = np.array([
        cities_states[i][0]
        for i in city_state_idx
    ], dtype=object)

    states = np.array([
        cities_states[i][1]
        for i in city_state_idx
    ], dtype=object)

    # ========================================================
    # TRANSACTION AMOUNT
    # ========================================================

    amount = np.exp(
        rng.normal(
            np.log(1400),
            0.9,
            n
        )
    )

    amount = np.clip(
        amount,
        20,
        100_000
    )

    # ========================================================
    # PAYMENT METHOD
    # ========================================================

    payment_method = rng.choice(
        payment_methods,
        n,
        p=[0.50, 0.18, 0.15, 0.10, 0.07]
    )

    # ========================================================
    # CARD NETWORK
    # Only applicable to card transactions
    # ========================================================

    card_network = np.array(
        [""] * n,
        dtype=object
    )

    card_mask = np.isin(
        payment_method,
        ["Credit Card", "Debit Card"]
    )

    card_network[card_mask] = rng.choice(
        card_networks,
        card_mask.sum(),
        p=[0.45, 0.35, 0.20]
    )

    # ========================================================
    # OTHER TRANSACTION ATTRIBUTES
    # ========================================================

    issuer_bank = rng.choice(
        banks,
        n,
        p=[
            0.18, 0.14, 0.13, 0.15, 0.08,
            0.07, 0.06, 0.06, 0.05, 0.08
        ]
    )

    payment_gateway = rng.choice(
        gateways,
        n,
        p=[0.30, 0.20, 0.22, 0.13, 0.15]
    )

    merchant_category = rng.choice(
        merchant_categories,
        n,
        p=[
            0.15, 0.08, 0.18, 0.10, 0.05,
            0.05, 0.08, 0.06, 0.08, 0.17
        ]
    )

    device_type = rng.choice(
        devices,
        n,
        p=[0.50, 0.32, 0.18]
    )

    # --------------------------------------------------------
    # OS Version based on device
    # --------------------------------------------------------

    os_version = np.empty(
        n,
        dtype=object
    )

    android_mask = device_type == "Android"
    ios_mask = device_type == "iOS"
    web_mask = device_type == "Web"

    os_version[android_mask] = rng.choice(
        ["Android 13", "Android 14", "Android 15"],
        android_mask.sum()
    )

    os_version[ios_mask] = rng.choice(
        ["iOS 17", "iOS 18"],
        ios_mask.sum()
    )

    os_version[web_mask] = rng.choice(
        ["Windows 11", "macOS 15"],
        web_mask.sum()
    )

    app_version = rng.choice(
        app_versions,
        n
    )

    network_type = rng.choice(
        networks,
        n,
        p=[0.60, 0.30, 0.10]
    )

    retry_count = rng.choice(
        [0, 1, 2, 3],
        n,
        p=[0.72, 0.18, 0.07, 0.03]
    )

    is_international = rng.choice(
        ["N", "Y"],
        n,
        p=[0.93, 0.07]
    )

    # ========================================================
    # PRE-TRANSACTION CUSTOMER HISTORY
    # ========================================================

    account_age_days = (
        account_age_days_master[
            customer_idx
        ]
    )

    past_txn_count_30d = (
        past_txn_count_master[
            customer_idx
        ]
    )

    past_failure_rate_30d = (
        past_failure_rate_master[
            customer_idx
        ]
    )

    avg_txn_amount_30d = (
        avg_txn_amount_master[
            customer_idx
        ]
    )

    account_balance_bucket = (
        account_balance_master[
            customer_idx
        ]
    )

    # ========================================================
    # FIRST TIME USER
    # ========================================================

    first_time_probability = np.where(
        account_age_days <= 90,
        0.35,
        0.08
    )

    is_first_time_user = np.where(
        rng.random(n) < first_time_probability,
        "Y",
        "N"
    )

    # ========================================================
    # CREATE LATENT FAILURE RISK
    #
    # IMPORTANT:
    # This risk is used to CREATE the target variable.
    # It is NOT saved in the CSV.
    # ========================================================

    transaction_hour = txn_dt.hour.to_numpy()

    peak_hour = np.isin(
        transaction_hour,
        [18, 19, 20, 21, 22]
    )

    # --------------------------------------------------------
    # Gateway risk
    # --------------------------------------------------------

    gateway_risk = np.select(
        [
            payment_gateway == "Razorpay",
            payment_gateway == "Cashfree",
            payment_gateway == "PayU",
            payment_gateway == "BillDesk",
            payment_gateway == "CCAvenue"
        ],
        [
            0.00,
            0.20,
            0.12,
            0.30,
            0.08
        ],
        default=0.0
    )

    # --------------------------------------------------------
    # Payment method risk
    # --------------------------------------------------------

    method_risk = np.select(
        [
            payment_method == "UPI",
            payment_method == "Credit Card",
            payment_method == "Debit Card",
            payment_method == "Net Banking",
            payment_method == "Wallet"
        ],
        [
            0.00,
            0.05,
            0.08,
            0.18,
            0.12
        ],
        default=0.0
    )

    # --------------------------------------------------------
    # Balance risk
    # --------------------------------------------------------

    balance_risk = np.select(
        [
            account_balance_bucket == "Low",
            account_balance_bucket == "Medium",
            account_balance_bucket == "High"
        ],
        [
            0.70,
            0.15,
            -0.20
        ],
        default=0
    )

    # --------------------------------------------------------
    # Current amount vs historical average
    # --------------------------------------------------------

    amount_ratio = (
        amount
        / np.maximum(
            avg_txn_amount_30d,
            1
        )
    )

    amount_risk = (
        np.clip(
            np.log1p(
                amount_ratio - 0.5
            ),
            -0.2,
            1.6
        )
        * 0.35
    )

    # --------------------------------------------------------
    # Combined latent score
    # --------------------------------------------------------

    raw_score = (
        -2.0

        + 1.9 * past_failure_rate_30d

        + 0.18 * retry_count

        + 0.40 * peak_hour

        + 0.60 * (
            network_type == "3G"
        )

        + 0.45 * (
            is_international == "Y"
        )

        + 0.55 * (
            is_first_time_user == "Y"
        )

        + gateway_risk

        + method_risk

        + balance_risk

        + amount_risk

        + 0.30 * (
            (device_type == "Web")
            & (transaction_hour < 7)
        )

        + 0.25 * (
            os_version == "Android 15"
        )
    )

    # ========================================================
    # CALIBRATE FAILURE RATE
    #
    # This keeps failure rate close to TARGET_FAILURE_RATE.
    # ========================================================

    uniforms = rng.random(n)

    lower = -6.0
    upper = 0.0

    for _ in range(50):

        mid = (
            lower + upper
        ) / 2

        probability = (
            1
            / (
                1
                + np.exp(
                    -(
                        raw_score
                        + (
                            mid + 1.0
                        )
                    )
                )
            )
        )

        estimated_rate = np.mean(
            uniforms < probability
        )

        if estimated_rate > target_failure_rate:
            upper = mid
        else:
            lower = mid

    intercept_adjustment = (
        lower + upper
    ) / 2

    final_probability = (
        1
        / (
            1
            + np.exp(
                -(
                    raw_score
                    + (
                        intercept_adjustment
                        + 1.0
                    )
                )
            )
        )
    )

    failed = (
        uniforms < final_probability
    )

    payment_status = np.where(
        failed,
        "Failed",
        "Success"
    )

    # ========================================================
    # FAILURE REASON
    # ========================================================

    failure_reason = np.array(
        [""] * n,
        dtype=object
    )

    response_code = np.array(
        ["00"] * n,
        dtype=object
    )

    reason_code_map = {
        "Insufficient funds": "51",
        "Bank server down": "96",
        "Timeout": "91",
        "Wrong PIN": "55",
        "OTP failure": "70",
        "Fraud block": "59",
        "Limit exceeded": "61"
    }

    failed_idx = np.where(
        failed
    )[0]

    # --------------------------------------------------------
    # Reason selection
    # --------------------------------------------------------

    for i in failed_idx:

        if account_balance_bucket[i] == "Low":

            reason_distribution = {
                "Insufficient funds": 0.40,
                "Bank server down": 0.10,
                "Timeout": 0.15,
                "Wrong PIN": 0.08,
                "OTP failure": 0.08,
                "Fraud block": 0.10,
                "Limit exceeded": 0.09
            }

        elif retry_count[i] >= 2:

            reason_distribution = {
                "Insufficient funds": 0.15,
                "Bank server down": 0.20,
                "Timeout": 0.28,
                "Wrong PIN": 0.07,
                "OTP failure": 0.08,
                "Fraud block": 0.12,
                "Limit exceeded": 0.10
            }

        elif is_international[i] == "Y":

            reason_distribution = {
                "Insufficient funds": 0.12,
                "Bank server down": 0.14,
                "Timeout": 0.16,
                "Wrong PIN": 0.05,
                "OTP failure": 0.08,
                "Fraud block": 0.30,
                "Limit exceeded": 0.15
            }

        else:

            reason_distribution = {
                "Insufficient funds": 0.18,
                "Bank server down": 0.16,
                "Timeout": 0.17,
                "Wrong PIN": 0.10,
                "OTP failure": 0.12,
                "Fraud block": 0.16,
                "Limit exceeded": 0.11
            }

        reasons = list(
            reason_distribution.keys()
        )

        probabilities = np.array(
            list(
                reason_distribution.values()
            )
        )

        probabilities = (
            probabilities
            / probabilities.sum()
        )

        chosen_reason = rng.choice(
            reasons,
            p=probabilities
        )

        failure_reason[i] = (
            chosen_reason
        )

        response_code[i] = (
            reason_code_map[
                chosen_reason
            ]
        )

    # ========================================================
    # PROCESSING TIME
    # ========================================================

    processing_time_ms = rng.lognormal(
        mean=np.log(750),
        sigma=0.50,
        size=n
    )

    processing_time_ms += (
        failed.astype(int)
        * rng.normal(
            200,
            120,
            n
        )
    )

    processing_time_ms = np.clip(
        processing_time_ms,
        50,
        12_000
    )

    processing_time_ms = (
        processing_time_ms
        .round()
        .astype(int)
    )

    # ========================================================
    # SYNTHETIC PII
    # ========================================================

    selected_names = (
        customer_master_names[
            customer_idx
        ].copy()
    )

    # --------------------------------------------------------
    # Email
    # --------------------------------------------------------

    email_domains = [
        "gmail.com",
        "outlook.com",
        "yahoo.com",
        "proton.me"
    ]

    emails = np.array([
        (
            re.sub(
                r"\s+",
                ".",
                str(name).lower()
            )
            + f"{rng.integers(1, 999)}@"
            + rng.choice(email_domains)
        )
        for name in selected_names
    ], dtype=object)

    # --------------------------------------------------------
    # Phone
    # --------------------------------------------------------

    valid_phones = np.array([
        str(
            rng.integers(
                6000000000,
                9999999999
            )
        )
        for _ in range(n)
    ], dtype=object)

    phones = valid_phones.copy()

    # --------------------------------------------------------
    # PAN
    # --------------------------------------------------------

    alphabet = np.array(
        list(
            "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        )
    )

    pans = []

    for _ in range(n):

        first_five = "".join(
            rng.choice(
                alphabet,
                5
            )
        )

        four_digits = (
            f"{rng.integers(0, 10000):04d}"
        )

        last_letter = rng.choice(
            alphabet
        )

        pans.append(
            first_five
            + four_digits
            + last_letter
        )

    pans = np.array(
        pans,
        dtype=object
    )

    # --------------------------------------------------------
    # Aadhaar last 4 only
    # --------------------------------------------------------

    aadhaar_last4 = np.array([
        f"{rng.integers(0, 10000):04d}"
        for _ in range(n)
    ], dtype=object)

    # --------------------------------------------------------
    # Masked card number
    # --------------------------------------------------------

    card_number_masked = np.array(
        [""] * n,
        dtype=object
    )

    for i in np.where(
        card_mask
    )[0]:

        last_four = (
            f"{rng.integers(1000, 10000)}"
        )

        if card_network[i] == "Visa":

            first_four = (
                f"{rng.integers(4000, 5000)}"
            )

        elif card_network[i] == "Mastercard":

            first_four = (
                f"{rng.integers(5100, 5600)}"
            )

        else:

            first_four = str(
                rng.choice(
                    [6081, 6521, 5085]
                )
            )

        card_number_masked[i] = (
            first_four
            + "XXXXXXXX"
            + last_four
        )

    # --------------------------------------------------------
    # UPI ID
    # --------------------------------------------------------

    upi_bank_codes = {
        "HDFC Bank": "hdfc",
        "ICICI Bank": "icici",
        "Axis Bank": "axis",
        "SBI": "sbi",
        "Kotak Mahindra Bank": "kotak",
        "Yes Bank": "yesbank",
        "IndusInd Bank": "indus",
        "IDFC First Bank": "idfc",
        "Federal Bank": "federal",
        "Bank of Baroda": "bob"
    }

    upi_id = np.array(
        [""] * n,
        dtype=object
    )

    upi_indexes = np.where(
        payment_method == "UPI"
    )[0]

    for i in upi_indexes:

        username = re.sub(
            r"[^a-z0-9]",
            "",
            selected_names[i].lower()
        )

        bank_code = upi_bank_codes.get(
            issuer_bank[i],
            "bank"
        )

        upi_id[i] = (
            f"{username[:12]}"
            f"{rng.integers(10, 99)}"
            f"@{bank_code}"
        )

    # --------------------------------------------------------
    # IPv4
    # --------------------------------------------------------

    ip_address = np.array([
        (
            f"{rng.integers(1, 223)}."
            f"{rng.integers(0, 256)}."
            f"{rng.integers(0, 256)}."
            f"{rng.integers(1, 255)}"
        )
        for _ in range(n)
    ], dtype=object)

    # --------------------------------------------------------
    # Pincode
    # --------------------------------------------------------

    billing_pincode = np.array([
        f"{rng.integers(100000, 1_000_000):06d}"
        for _ in range(n)
    ], dtype=object)

    # ========================================================
    # DIRTY DATA
    # ========================================================

    # --------------------------------------------------------
    # 1. Missing values
    #    ~4% each
    # --------------------------------------------------------

    missing_columns = [
        emails,
        phones,
        cities,
        device_type
    ]

    for column in missing_columns:

        missing_idx = rng.choice(
            n,
            size=round(0.04 * n),
            replace=False
        )

        column[missing_idx] = None

    # --------------------------------------------------------
    # 2. Invalid email addresses
    # --------------------------------------------------------

    email_dirty_idx = rng.choice(
        n,
        size=round(0.025 * n),
        replace=False
    )

    invalid_emails = [
        "missing-at-domain",
        "user@@gmail.com",
        "john..doe@gmail.com",
        "test@invalid",
        "abc@tempmail",
        "abc gmail.com"
    ]

    for i in email_dirty_idx:

        emails[i] = rng.choice(
            invalid_emails
        )

    # --------------------------------------------------------
    # 3. Dirty phone numbers
    # --------------------------------------------------------

    phone_dirty_idx = rng.choice(
        n,
        size=round(0.04 * n),
        replace=False
    )

    for i in phone_dirty_idx:

        original_phone = (
            valid_phones[i]
        )

        phones[i] = rng.choice([
            "+91 "
            + original_phone[:5]
            + " "
            + original_phone[5:],

            "+91-"
            + original_phone,

            original_phone[1:],  # 9 digits

            "98AB"
            + original_phone[4:],

            "1234567890"
        ])

    # --------------------------------------------------------
    # 4. Dirty PAN numbers
    # --------------------------------------------------------

    pan_dirty_idx = rng.choice(
        n,
        size=round(0.025 * n),
        replace=False
    )

    for i in pan_dirty_idx:

        pans[i] = rng.choice([
            pans[i].lower(),
            pans[i][:4] + "12345",
            "12345ABCDE",
            "ABCDE1234",
            "AB12C3456D",
            "AAAAA-9999-A"
        ])

    # --------------------------------------------------------
    # 5. Dirty customer names
    # --------------------------------------------------------

    name_dirty_idx = rng.choice(
        n,
        size=round(0.05 * n),
        replace=False
    )

    for i in name_dirty_idx:

        name = str(
            selected_names[i]
        )

        selected_names[i] = rng.choice([
            "  "
            + name.lower()
            + " ",

            " "
            + name.upper()
            + "###",

            name.replace(
                " ",
                "   "
            ),

            name
            + " !!",

            name.title()
        ])

    # ========================================================
    # CITY STANDARDIZATION PROBLEMS
    # ========================================================

    city_variants = {
        "Ahmedabad": [
            "Ahmedabad",
            "ahmedabad",
            "Ahmdabad",
            "Ahemdabad"
        ],
        "Mumbai": [
            "Mumbai",
            "mumbai",
            "Bombay"
        ],
        "Bengaluru": [
            "Bengaluru",
            "Bangalore",
            "bangalore"
        ],
        "Gurugram": [
            "Gurugram",
            "Gurgaon",
            "gurgaon"
        ],
        "Delhi": [
            "Delhi",
            "delhi",
            "New Delhi"
        ]
    }

    for canonical, variants in city_variants.items():

        matching_idx = np.where(
            cities == canonical
        )[0]

        if len(matching_idx) > 0:

            number_to_dirty = max(
                1,
                int(0.20 * len(matching_idx))
            )

            dirty_idx = rng.choice(
                matching_idx,
                size=number_to_dirty,
                replace=False
            )

            for i in dirty_idx:

                cities[i] = rng.choice(
                    variants
                )

    # ========================================================
    # STATE STANDARDIZATION PROBLEMS
    # ========================================================

    state_variants = {
        "Gujarat": [
            "Gujarat",
            "gujarat",
            "Gujrat"
        ],
        "Maharashtra": [
            "Maharashtra",
            "maharashtra",
            "Maharastra"
        ],
        "Karnataka": [
            "Karnataka",
            "karnataka",
            "Karnatka"
        ],
        "Telangana": [
            "Telangana",
            "telangana",
            "Telengana"
        ],
        "Tamil Nadu": [
            "Tamil Nadu",
            "tamil nadu",
            "Tamilnadu"
        ]
    }

    for canonical, variants in state_variants.items():

        matching_idx = np.where(
            states == canonical
        )[0]

        if len(matching_idx) > 0:

            number_to_dirty = max(
                1,
                int(0.15 * len(matching_idx))
            )

            dirty_idx = rng.choice(
                matching_idx,
                size=number_to_dirty,
                replace=False
            )

            for i in dirty_idx:

                states[i] = rng.choice(
                    variants
                )

    # ========================================================
    # BANK NAME INCONSISTENCY
    # ========================================================

    bank_variants = {
        "HDFC Bank": [
            "HDFC Bank",
            "hdfc",
            "Hdfc",
            "HDFC BANK"
        ],
        "ICICI Bank": [
            "ICICI Bank",
            "icici",
            "Icici",
            "ICICI BANK"
        ],
        "Axis Bank": [
            "Axis Bank",
            "axis",
            "AXIS BANK"
        ],
        "SBI": [
            "SBI",
            "sbi",
            "State Bank of India"
        ],
        "Kotak Mahindra Bank": [
            "Kotak Mahindra Bank",
            "kotak",
            "KOTAK"
        ]
    }

    bank_dirty_mask = (
        rng.random(n) < 0.08
    )

    for canonical, variants in bank_variants.items():

        idx = np.where(
            (issuer_bank == canonical)
            & bank_dirty_mask
        )[0]

        for i in idx:

            issuer_bank[i] = rng.choice(
                variants
            )

    # ========================================================
    # GATEWAY NAME INCONSISTENCY
    # ========================================================

    gateway_variants = {
        "Razorpay": [
            "Razorpay",
            "razorpay",
            "Razor Pay",
            "RAZORPAY"
        ],
        "Cashfree": [
            "Cashfree",
            "cashfree",
            "Cash Free",
            "CASHFREE"
        ],
        "PayU": [
            "PayU",
            "payu",
            "Pay U"
        ]
    }

    for canonical, variants in gateway_variants.items():

        idx = np.where(
            payment_gateway == canonical
        )[0]

        if len(idx) > 0:

            dirty_count = max(
                1,
                int(0.10 * len(idx))
            )

            dirty_idx = rng.choice(
                idx,
                size=dirty_count,
                replace=False
            )

            for i in dirty_idx:

                payment_gateway[i] = (
                    rng.choice(variants)
                )

    # ========================================================
    # AMOUNT ERRORS
    # ========================================================

    # Zero amounts
    zero_idx = rng.choice(
        n,
        size=round(0.007 * n),
        replace=False
    )

    amount[zero_idx] = 0

    # Negative amounts
    remaining_idx = np.setdiff1d(
        np.arange(n),
        zero_idx
    )

    negative_idx = rng.choice(
        remaining_idx,
        size=round(0.007 * n),
        replace=False
    )

    amount[negative_idx] = (
        -np.abs(
            amount[negative_idx]
        )
    )

    # Extreme outliers
    remaining_idx = np.setdiff1d(
        remaining_idx,
        negative_idx
    )

    outlier_idx = rng.choice(
        remaining_idx,
        size=round(0.004 * n),
        replace=False
    )

    amount[outlier_idx] = rng.choice(
        [
            250_000,
            500_000,
            999_999,
            1_500_000
        ],
        size=len(outlier_idx)
    )

    # ========================================================
    # MIXED DATE FORMATS
    # ========================================================

    date_formats = [
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y %H:%M",
        "%m-%d-%Y %H:%M",
        "%d-%b-%Y %H:%M:%S",
        "%Y/%m/%d %H:%M"
    ]

    txn_timestamp = np.empty(
        n,
        dtype=object
    )

    for i, dt in enumerate(txn_dt):

        selected_format = rng.choice(
            date_formats,
            p=[
                0.45,
                0.20,
                0.15,
                0.10,
                0.10
            ]
        )

        txn_timestamp[i] = (
            pd.Timestamp(dt)
            .strftime(
                selected_format
            )
        )

    # ========================================================
    # CARD NUMBER MASK ERRORS
    # ========================================================

    card_indexes = np.where(
        card_number_masked != ""
    )[0]

    if len(card_indexes) > 0:

        corrupt_count = max(
            1,
            round(0.01 * len(card_indexes))
        )

        corrupt_idx = rng.choice(
            card_indexes,
            size=corrupt_count,
            replace=False
        )

        for i in corrupt_idx:

            card_number_masked[i] = rng.choice([
                card_number_masked[i][:-1],
                card_number_masked[i].replace(
                    "X",
                    "0",
                    1
                ),
                "4111XXXXXXXX111",
                "4111XXXX1111XXXX"
            ])

    # ========================================================
    # INVALID IP ADDRESSES
    # ========================================================

    invalid_ip_idx = rng.choice(
        n,
        size=round(0.01 * n),
        replace=False
    )

    invalid_ips = [
        "999.12.1.1",
        "192.168.1",
        "10.0.0.256",
        "abc.def.1.2"
    ]

    for i in invalid_ip_idx:

        ip_address[i] = (
            rng.choice(
                invalid_ips
            )
        )

    # ========================================================
    # INVALID PINCODES
    # ========================================================

    invalid_pin_idx = rng.choice(
        n,
        size=round(0.01 * n),
        replace=False
    )

    invalid_pincodes = [
        "3800A1",
        "12345",
        "1234567",
        "******"
    ]

    for i in invalid_pin_idx:

        billing_pincode[i] = (
            rng.choice(
                invalid_pincodes
            )
        )

    # ========================================================
    # LOGICAL INCONSISTENCIES
    # ========================================================

    failed_rows = np.where(
        payment_status == "Failed"
    )[0]

    success_rows = np.where(
        payment_status == "Success"
    )[0]

    # Failed transactions with blank failure_reason
    if len(failed_rows) > 0:

        blank_reason_idx = rng.choice(
            failed_rows,
            size=max(
                1,
                round(0.02 * len(failed_rows))
            ),
            replace=False
        )

        failure_reason[
            blank_reason_idx
        ] = ""

    # Successful transactions with a failure reason
    if len(success_rows) > 0:

        wrong_reason_idx = rng.choice(
            success_rows,
            size=max(
                1,
                round(0.015 * len(success_rows))
            ),
            replace=False
        )

        random_reasons = list(
            reason_code_map.keys()
        )

        random_codes = list(
            reason_code_map.values()
        )

        failure_reason[
            wrong_reason_idx
        ] = rng.choice(
            random_reasons,
            size=len(wrong_reason_idx)
        )

        response_code[
            wrong_reason_idx
        ] = rng.choice(
            random_codes,
            size=len(wrong_reason_idx)
        )

    # ========================================================
    # PROCESSING TIME OUTLIERS
    # ========================================================

    latency_outlier_idx = rng.choice(
        n,
        size=round(0.006 * n),
        replace=False
    )

    processing_time_ms[
        latency_outlier_idx
    ] = rng.choice(
        [
            15_000,
            30_000,
            60_000
        ],
        size=len(
            latency_outlier_idx
        )
    )

    # ========================================================
    # BUILD FINAL DATAFRAME
    # ========================================================

    df = pd.DataFrame({

        # ----------------------------------------------------
        # IDENTIFIERS
        # ----------------------------------------------------

        "transaction_id": [
            f"TXN{10000000 + i}"
            for i in range(n)
        ],

        "customer_id": (
            customer_ids[
                customer_idx
            ]
        ),

        "merchant_id": (
            merchant_ids[
                merchant_idx
            ]
        ),

        # ----------------------------------------------------
        # PII
        # ----------------------------------------------------

        "customer_name": selected_names,

        "email": emails,

        "phone_number": phones,

        "dob": [
            dob_dates[idx]
            .strftime("%Y-%m-%d")
            for idx in customer_idx
        ],

        "pan_number": pans,

        "aadhaar_last4": (
            aadhaar_last4
        ),

        "card_number_masked": (
            card_number_masked
        ),

        "upi_id": upi_id,

        "ip_address": ip_address,

        "billing_pincode": (
            billing_pincode
        ),

        "city": cities,

        "state": states,

        # ----------------------------------------------------
        # TRANSACTION DATA
        # ----------------------------------------------------

        "txn_timestamp": (
            txn_timestamp
        ),

        "amount": np.round(
            amount,
            2
        ),

        "currency": np.array(
            ["INR"] * n
        ),

        "payment_method": (
            payment_method
        ),

        "card_network": (
            card_network
        ),

        "issuer_bank": (
            issuer_bank
        ),

        "payment_gateway": (
            payment_gateway
        ),

        "merchant_category": (
            merchant_category
        ),

        "device_type": (
            device_type
        ),

        "os_version": (
            os_version
        ),

        "app_version": (
            app_version
        ),

        "network_type": (
            network_type
        ),

        "retry_count": (
            retry_count
        ),

        "is_international": (
            is_international
        ),

        "is_first_time_user": (
            is_first_time_user
        ),

        # ----------------------------------------------------
        # CUSTOMER HISTORY
        # These are pre-transaction variables
        # ----------------------------------------------------

        "account_age_days": (
            account_age_days
        ),

        "past_txn_count_30d": (
            past_txn_count_30d
        ),

        "past_failure_rate_30d": np.round(
            past_failure_rate_30d,
            4
        ),

        "avg_txn_amount_30d": np.round(
            avg_txn_amount_30d,
            2
        ),

        "account_balance_bucket": (
            account_balance_bucket
        ),

        # ----------------------------------------------------
        # OUTCOME
        # ----------------------------------------------------

        "payment_status": (
            payment_status
        ),

        "failure_reason": (
            failure_reason
        ),

        "response_code": (
            response_code
        ),

        "processing_time_ms": (
            processing_time_ms
        )
    })

    # ========================================================
    # ADD DUPLICATE TRANSACTION IDS
    # ~2.5%
    #
    # We only duplicate the ID.
    # Other values remain different, allowing Alteryx to
    # detect duplicate keys without simply removing
    # identical rows.
    # ========================================================

    duplicate_count = round(
        0.025 * n
    )

    duplicate_targets = rng.choice(
        n,
        size=duplicate_count,
        replace=False
    )

    all_rows = np.arange(n)

    for target in duplicate_targets:

        possible_sources = (
            all_rows[
                all_rows != target
            ]
        )

        source = int(
            rng.choice(
                possible_sources
            )
        )

        df.loc[
            target,
            "transaction_id"
        ] = df.loc[
            source,
            "transaction_id"
        ]

    # ========================================================
    # SHUFFLE ROWS
    # ========================================================

    df = df.sample(
        frac=1,
        random_state=seed
    ).reset_index(
        drop=True
    )

    return df


# ============================================================
# GENERATE DATA
# ============================================================

df = generate_dataset(
    n=N_ROWS,
    seed=SEED,
    target_failure_rate=TARGET_FAILURE_RATE
)


# ============================================================
# SAVE CSV
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# QUALITY CHECKS
# ============================================================

print("=" * 60)
print("DATASET CREATED")
print("=" * 60)

print(f"Rows              : {len(df):,}")
print(f"Columns           : {len(df.columns)}")
print(
    f"Failure rate      : "
    f"{(df['payment_status'] == 'Failed').mean():.2%}"
)

print(
    f"Unique transaction IDs : "
    f"{df['transaction_id'].nunique():,}"
)

print(
    f"Duplicate transaction ID rows : "
    f"{df['transaction_id'].duplicated(keep=False).sum():,}"
)

print("\nMissing values:")
print(
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .head(10)
      .mul(100)
      .round(2)
      .astype(str)
      .add("%")
)

print("\nPayment status:")
print(
    df["payment_status"]
      .value_counts()
)

print("\nFailure reasons:")
print(
    df.loc[
        df["payment_status"] == "Failed",
        "failure_reason"
    ]
    .value_counts(dropna=False)
)

print("\nPayment methods:")
print(
    df["payment_method"]
      .value_counts()
)

print("\nGateways:")
print(
    df["payment_gateway"]
      .value_counts()
)

print("\nSample rows:")
print(
    df.head(5).to_string(
        index=False
    )
)

print("\nSaved to:")
print(OUTPUT_FILE)

DATASET CREATED
Rows              : 10,000
Columns           : 39
Failure rate      : 10.01%
Unique transaction IDs : 9,754
Duplicate transaction ID rows : 486

Missing values:
city                   4.0%
email                 3.91%
phone_number          3.88%
merchant_id            0.0%
customer_id            0.0%
customer_name          0.0%
transaction_id         0.0%
dob                    0.0%
pan_number             0.0%
card_number_masked     0.0%
dtype: object

Payment status:
payment_status
Success    8999
Failed     1001
Name: count, dtype: int64

Failure reasons:
failure_reason
Insufficient funds    226
Timeout               178
Bank server down      142
Limit exceeded        127
Fraud block           125
Wrong PIN              99
OTP failure            84
                       20
Name: count, dtype: int64

Payment methods:
payment_method
UPI            4978
Credit Card    1886
Debit Card     1463
Net Banking     972
Wallet          701
Name: count, dtype: int64

Gateways:
pa